In [1]:
# ==============================================================================
# ENVIRONMENT SETUP AND GITHUB AUTHENTICATION WITHOUT GOOGLE DRIVE
# Objective: Configure GitHub authentication and clone or update the repository
# directly inside the temporary Google Colab environment.
# ==============================================================================

# Import os for file-system operations and working-directory management.
import os

# Import userdata to retrieve the GitHub token from Colab Secrets.
from google.colab import userdata


# ------------------------------------------------------------------------------
# 1. PARAMETER CONFIGURATION
# ------------------------------------------------------------------------------

# Define the GitHub account and repository names.
GITHUB_USER = "pascual-francisco"
GITHUB_REPO = "INF-8239-Ciencia-de-Datos-II"

# Retrieve the GitHub personal access token from Colab Secrets.
GITHUB_TOKEN = userdata.get("TOKEN")

# Define the repository location directly inside the Colab environment.
REPO_DIR = f"/content/{GITHUB_REPO}"

# Define the author information used in Git commits.
GIT_AUTHOR_NAME = "Francisco Pascual"
GIT_AUTHOR_EMAIL = "pascual.francisco@gmail.com"


# ------------------------------------------------------------------------------
# 2. VERIFY THE GITHUB TOKEN
# ------------------------------------------------------------------------------

# Stop execution if the TOKEN secret is unavailable.
if not GITHUB_TOKEN:
    raise ValueError(
        "The GitHub token was not found. "
        "Add a Colab Secret named TOKEN before running this cell."
    )


# ------------------------------------------------------------------------------
# 3. GLOBAL GIT CONFIGURATION
# ------------------------------------------------------------------------------

# Configure the author name used in Git commits.
!git config --global user.name "{GIT_AUTHOR_NAME}"

# Configure the author email used in Git commits.
!git config --global user.email "{GIT_AUTHOR_EMAIL}"

# Configure main as the default branch name.
!git config --global init.defaultBranch main


# ------------------------------------------------------------------------------
# 4. DEFINE THE GITHUB REPOSITORY URL
# ------------------------------------------------------------------------------

# Define the authenticated URL used to clone or update the repository.
AUTH_URL = (
    f"https://{GITHUB_USER}:{GITHUB_TOKEN}"
    f"@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
)

# Define a clean URL that does not contain the GitHub token.
CLEAN_URL = (
    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
)


# ------------------------------------------------------------------------------
# 5. CLONE OR UPDATE THE REPOSITORY
# ------------------------------------------------------------------------------

# Clone the repository when it is not available in the current Colab session.
if not os.path.exists(REPO_DIR):
    print(f"Cloning {GITHUB_REPO}...")

    # Clone the repository using the authenticated URL.
    !git clone "{AUTH_URL}" "{REPO_DIR}"

else:
    print("Repository already exists. Updating from the main branch...")

    # Navigate to the existing repository.
    %cd {REPO_DIR}

    # Retrieve and merge the latest changes using authenticated access.
    !git pull "{AUTH_URL}" main


# ------------------------------------------------------------------------------
# 6. REMOVE THE TOKEN FROM THE STORED REMOTE URL
# ------------------------------------------------------------------------------

# Navigate to the repository after cloning or updating it.
%cd {REPO_DIR}

# Store a clean GitHub URL as the permanent origin remote.
# This prevents the token from remaining visible in the Git configuration.
!git remote set-url origin "{CLEAN_URL}"


# ------------------------------------------------------------------------------
# 7. VERIFY THE ACTIVE ENVIRONMENT
# ------------------------------------------------------------------------------

# Display the active repository location.
print(f"\nEnvironment ready and located in:\n{os.getcwd()}")

# Display the configured Git remote without exposing the token.
print("\nConfigured Git remote:")
!git remote -v

# Display the current repository status.
print("\nCurrent Git status:")
!git status --short

Cloning INF-8239-Ciencia-de-Datos-II...
Cloning into '/content/INF-8239-Ciencia-de-Datos-II'...
remote: Enumerating objects: 829, done.
remote: Counting objects: 100% (259/259), done.
remote: Compressing objects: 100% (193/193), done.
remote: Total 829 (delta 147), reused 161 (delta 65), pack-reused 570 (from 1)
Receiving objects: 100% (829/829), 31.87 MiB | 20.80 MiB/s, done.
Resolving deltas: 100% (483/483), done.
/content/INF-8239-Ciencia-de-Datos-II

Environment ready and located in:
/content/INF-8239-Ciencia-de-Datos-II

Configured Git remote:
origin	https://github.com/pascual-francisco/INF-8239-Ciencia-de-Datos-II.git (fetch)
origin	https://github.com/pascual-francisco/INF-8239-Ciencia-de-Datos-II.git (push)

Current Git status:


In [9]:
# ==============================================================================
# CELL 01 — ENVIRONMENT SETUP AND IMPORT REQUIRED LIBRARIES
# Objective: Install missing dependencies (if needed) and import all libraries
# required for LAB06 embeddings and network analysis.
# ==============================================================================

# Install missing packages in Colab
try:
    from gensim.models import Word2Vec
except ModuleNotFoundError:
    !pip -q install gensim networkx
    from gensim.models import Word2Vec

# Standard library
from pathlib import Path
import os
import warnings

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Similarity metrics
from sklearn.metrics.pairwise import cosine_similarity

# Network analysis
import networkx as nx
from networkx.algorithms.community import (
    greedy_modularity_communities,
    modularity,
)

# Notebook display
from IPython.display import display

# Configuration
warnings.filterwarnings("ignore")

print("LAB06 environment ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 36.8 MB/s eta 0:00:00
LAB06 environment ready.


In [31]:
# ==============================================================================
# CELL 02 — PREPARE AND VERIFY THE NLP ENVIRONMENT
# Objective: Synchronize project dependencies and execute the test suite
# before training the Word2Vec model.
# ==============================================================================

# Move to the NLP project root directory
%cd /content/INF-8239-Ciencia-de-Datos-II/Unit_02_Natural_Language_Processing/nlp_sentiment_analysis

# Synchronize project dependencies
!uv sync

# Execute the automated test suite
!uv run pytest -q

/content/INF-8239-Ciencia-de-Datos-II/Unit_02_Natural_Language_Processing/nlp_sentiment_analysis
Resolved 64 packages in 1ms
Checked 63 packages in 1ms
......                                                                   [100%]
6 passed in 2.76s


In [32]:
# ==============================================================================
# CELL 03 — TRAIN WORD2VEC EMBEDDINGS
# Objective: Train a Word2Vec model using the corpus and inspect
# semantic similarity relationships for a selected word.
# ==============================================================================

# Train the Word2Vec model and inspect semantic neighbors
!uv run python scripts/embeddings_network.py --word servicio

Vocabulario: 37818 Cobertura: 1.0
Vecinos: [('sector', 0.7294051647186279), ('administración', 0.7137258648872375), ('ajustazobotella', 0.7115362286567688), ('psoemadrid', 0.6966636776924133), ('protección', 0.6817774176597595), ('cuenta_gonzalo_lópez_alba', 0.6773442625999451), ('sanitario', 0.6691912412643433), ('salario', 0.6547726988792419), ('barat', 0.6541843414306641), ('empresa', 0.6533589959144592)]


## Word2Vec Results

The model generated a vocabulary containing 37,818 unique tokens.

The embedding space achieved 100% token coverage, indicating that all corpus tokens were represented in the learned vocabulary.

For the selected word "servicio", the closest neighbors included sector, administración, protección, sanitario, salario, and empresa.

These results suggest contextual similarity based on word co-occurrence patterns in the corpus. Some unusual neighbors may be explained by corpus-specific terminology, noise, or limited contextual information. High similarity values indicate proximity in the embedding space but do not imply universal synonymy.

In [10]:
# ==============================================================================
# CELL 04 — EVALUATE THE IMPACT OF THE WINDOW PARAMETER
# Objective: Compare semantic neighbors before and after modifying the
# Word2Vec context window and analyze how the captured context changes.
# ==============================================================================

import os

os.environ.pop("MPLBACKEND", None)

!uv run python scripts/embeddings_network.py --word servicio

Vocabulario: 37818 Cobertura: 1.0
Vecinos: [('administración', 0.7211803793907166), ('protección', 0.7096176147460938), ('dependencia', 0.6997079849243164), ('sector', 0.6984333992004395), ('seguridad', 0.6973701119422913), ('transporte', 0.6845107078552246), ('deterioro', 0.6660933494567871), ('patrimonio', 0.6643491983413696), ('congelación', 0.6439675092697144), ('mínimo', 0.6432412266731262)]


## Hypothesis

Increasing the window size from 5 to 10 will allow the model to capture a broader context around each word, producing different semantic neighbors.

## Results

The vocabulary size remained unchanged at 37,818 tokens.

Token coverage remained at 100% because the minimum frequency threshold was not modified.

However, the nearest neighbors of the word "servicio" changed after increasing the context window. The original model emphasized nearby contextual terms such as sanitario, salario, and empresa, while the modified model produced neighbors such as seguridad, transporte, dependencia, and patrimonio.

## Interpretation

The larger window allowed Word2Vec to consider words appearing farther from the target term. As a result, the model captured broader contextual relationships rather than focusing only on local word co-occurrences.

The change in semantic neighbors indicates that the context window influences the type of linguistic information learned by the embedding model.

## Selected Value

The experiment was completed using window=10 and the change was documented for reproducibility.
``

In [11]:
# ==============================================================================
# CELL 05 — ANALYZE THE DEMONSTRATION NETWORK
# Objective: Execute the Zachary Karate Club network analysis and generate
# centrality metrics, community detection results, and network visualizations.
# ==============================================================================

# Run embeddings and network analysis
!env -u MPLBACKEND uv run python scripts/embeddings_network.py --word servicio --network-demo

Vocabulario: 37818 Cobertura: 1.0
Vecinos: [('administración', 0.7211803793907166), ('protección', 0.7096176147460938), ('dependencia', 0.6997079849243164), ('sector', 0.6984333992004395), ('seguridad', 0.6973701119422913), ('transporte', 0.6845107078552246), ('deterioro', 0.6660933494567871), ('patrimonio', 0.6643491983413696), ('congelación', 0.6439675092697144), ('mínimo', 0.6432412266731262)]
Nodos: 34 Aristas: 78
Comunidades: 3 Modularidad: 0.411


## Network Structure Results

The Zachary Karate Club network contains 34 nodes and 78 edges.

Community detection identified 3 structural communities.

The modularity score was 0.411, indicating a moderate separation between
communities. Members within the same community tend to be more densely
connected to each other than to members of other communities.

These results describe structural properties of the network only and
should not be interpreted as evidence of influence, leadership, authority,
or causal relationships.

In [ ]:
# ==============================================================================
# CELL 06 — COMPARE NETWORK CENTRALITIES
# Objective: Compare degree centrality, betweenness centrality,
# and PageRank while respecting the interpretation limits of
# network analysis.
# ==============================================================================

## Degree Centrality

Degree centrality measures the number of direct connections associated
with a node.

A node with high degree centrality has many local connections within
the network. However, this does not necessarily mean that the node
connects different regions of the graph.

## Betweenness Centrality

Betweenness centrality measures how often a node appears in the
shortest paths connecting other nodes.

A node with moderate degree but high betweenness may act as a
structural bridge between communities and facilitate connectivity
between otherwise separated regions of the network.

## PageRank

PageRank evaluates both the quantity and quality of connections.

A node with high PageRank receives connections from nodes that are
also structurally important within the network.

## Communities

The detected communities represent groups identified by the community
detection algorithm in this specific network.

These communities should be interpreted as structural groupings and
not as essential social identities.

## Interpretation Limits

Network metrics describe structural properties only.

A node with the highest degree, betweenness, or PageRank should not
be described as the most influential person, leader, or authority
figure in the network.

The analysis only supports statements about connectivity patterns,
bridging roles, and structural importance according to the selected
metric.

In [13]:
# ==============================================================================
# CELL 07 — TESTING AND REPRODUCIBILITY
# Objective: Verify that the embeddings and network analysis components
# work correctly and prepare the project for version control.
# ==============================================================================

# Execute the LAB06 automated tests
!uv run pytest test/test_embeddings_network.py -q

# Review repository status before committing changes
!git status

..                                                                       [100%]
2 passed in 2.48s
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   pyproject.toml
	modified:   scripts/embeddings_network.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	models/word2vec.model
	reports/centralities.csv
	reports/network.png
	reports/social_network.graphml

no changes added to commit (use "git add" and/or "git commit -a")


In [ ]:
# ==============================================================================
# CELL 08 — PORTABILITY AND INTERPRETATION
# Objective: Document reproducibility considerations, portability
# limitations, and responsible interpretation of embeddings and
# network analysis results.
# ==============================================================================

## Portability

The trained Word2Vec model and all generated reports were saved using
relative project paths, improving portability across different systems.

The model was trained using a fixed random seed and workers=1. These
settings reduce variability between executions, although exact binary
reproducibility across different operating systems and hardware
configurations is not guaranteed.

For real-world social network datasets, documentation should include
the definition of nodes, edges, edge direction, edge weight, time
period, and data collection consent.

Network visualizations intended for publication should not include
personally identifiable information without explicit authorization.

## Resultados de Embeddings

A Word2Vec model was trained using the corpus. The selected word was
"servicio". The nearest neighbors included administración,
protección, dependencia, sector, seguridad, transporte,
deterioro, patrimonio, congelación, and mínimo.

These words reflect contextual similarity learned from the corpus and
should not be interpreted as universal synonyms.

## Evidencia de Cobertura

The model generated a vocabulary containing 37,818 unique tokens.

The token coverage was 1.0 (100%), indicating that all corpus tokens
were represented in the learned vocabulary.

## Resultado Estructural de la Red

The Zachary Karate Club network contained 34 nodes and 78 edges.

Community detection identified 3 structural communities with a
modularity score of 0.411, indicating a moderate level of separation
between groups.

## Dos Métricas Comparadas

Degree centrality measures the number of direct connections of a node.

Betweenness centrality measures how frequently a node appears on the
shortest paths connecting other nodes and can indicate a potential
bridging role between communities.

## Interpretación Permitida

The detected communities represent structural groupings identified by
the algorithm. Nodes with high degree centrality have many direct
connections, while nodes with high betweenness centrality may serve as
structural bridges between regions of the network.

## Interpretación que NO Puede Sostenerse

The analysis does not demonstrate that a node is the most influential,
most powerful, most important, or a leader within the network.

Likewise, the detected communities should not be interpreted as fixed
social identities or causal relationships.

## Siguiente Experimento

A future experiment could evaluate the effect of different Word2Vec
parameters such as min_count, vector_size, or epochs. Additional
network experiments could compare alternative community detection
algorithms or analyze weighted and directed networks.

In [14]:
%%writefile /content/INF-8239-Ciencia-de-Datos-II/CHANGELOG.md
# Changelog

## [1.7.0] - 2026-10-07
### Added
- Completed LAB 06.
- Completed DEMO 01.
- Added TF-IDF pipelines using DummyClassifier, Complement Naive Bayes, and Logistic Regression.
- Added model metrics, confusion matrix, error analysis, and a reusable Joblib pipeline.
- Added automated model tests and an interactive Streamlit sentiment application.
- Added cloud-compatible dependencies through `requirements-cloud.txt`.

### Changed
- Updated project imports to use the existing `src` package.
- Updated Exercise 03 documentation to integrate LAB04 and LAB05.

### Validated
- Verified model prediction and pipeline persistence.
- Verified that the Streamlit application starts successfully in a clean environment.

All notable changes to this repository will be documented in this file.
## [1.6.0] - 2026-10-02

### Added
- Added the complete LAB04 workflow for selecting, downloading, reconstructing, auditing, and validating a public Spanish sentiment corpus.
- Added dataset candidate evaluation based on relevance, license, representativeness, quality, reproducibility, and risk.
- Added SHA-256 verification for the downloaded corpus.
- Added reconstruction of encoded reviews using the provided word dictionary and metadata.
- Added a processed three-class dataset containing positive, neutral, and negative sentiment labels.
- Added automated data-contract tests for required columns and non-empty text values.
- Added dataset documentation, audit results, and Colab portability configuration.
- Updated the repository-level `.gitignore` to exclude local `.env` files and raw and processed datasets from nested projects.

## [1.5.0] - 2026-09-24
### Added
- Created the folder structures for the NLP and Computer Vision projects.
- Created the folder structure for `Unit_02_Natural_Language_Processing/`:
- `app/`
- `data/raw/`
- `data/processed/`
- `docs/`
- `notebooks/`
- `reports/figures/`
- `scripts/`
- `src/`
- `tests/`

## [1.4.0] - 2026-09-24
### Added
- Added additional Green AI visualization outputs under `reports/figures/`.
- Added `model_comparison_f1.png` to compare F1 Macro performance across all evaluated models.
- Added `training_time_comparison.png` to compare median training times.
- Added `inference_time_comparison.png` to compare prediction latency across models.
- Added `model_size_comparison.png` to compare serialized model sizes.
- Added `green_ai_dashboard.png` consolidating F1, training time, inference time, and model size into a single report figure.
- Preserved existing required figures (`pareto.png` and `tsne_two_seeds.png`) and organized all report visualizations in `reports/figures/`.
- Added Green AI benchmarking workflow for U01.LAB03.
- Added six model configurations:
  - Logistic Regression
  - SVM (C=1)
  - SVM (C=10)
  - Random Forest (100 trees)
  - Random Forest (300 trees)
  - Histogram Gradient Boosting
- Added PCA dimensionality-reduction analysis.
- Added t-SNE visualizations using two different random seeds.
- Added Pareto frontier analysis for performance-cost trade-off evaluation.
- Added serialized model size and inference-time benchmarking.
- Added automated Pareto validation tests (`tests/test_green.py`).

### Generated
- Generated `reports/green_ai_results.csv`.
- Generated `reports/figures/pareto.png`.
- Generated `reports/figures/tsne_two_seeds.png`.
- Generated serialized model artifacts under `reports/models/`.

### Documentation
- Added Green AI interpretation and model-selection discussion.
- Added execution environment registration for reproducibility.

## [1.3.0] - 2026-09-22
- Added U01.LAB03 notebook

## [1.2.0] - 2026-09-22

### Added
- Completed the U01.LAB02 dataset audit and baseline modeling workflow.
- Defined the `Class` target and removed potential leakage variables.
- Added preprocessing for numerical and categorical features.
- Compared the `DummyClassifier` baseline against the RBF SVM.
- Added automated dataset-contract tests.
- Added dataset documentation, evaluation results, and conclusions.

### Results
- Dummy baseline F1-Macro: `0.3475`
- SVM F1-Macro: `0.6921`
- SVM accuracy: `0.6926`
- Positive-class recall: `0.6098`
- Dataset contract tests: `3 passed`

### Status
- U01.LAB02 guided workflow completed.
- The justified model adaptation and final Exercise 01 comparison remain pending.

## [1.1.0] - 2026-09-22

### Added
- Started U01.LAB02 public dataset search, selection, and audit workflow.
- Created the dataset profile and documented the analytical problem.
- Compared the Diabetic Retinopathy Debrecen and Dry Bean datasets.
- Applied the dataset acceptance criteria to both candidates.
- Documented dataset sources, targets, classes, dimensions, missing values, and potential leakage risks.
- Added reproducible access to the UCI Machine Learning Repository through the `ucimlrepo` API.
- Added the reusable `download_uci_dataset()` function in `data/data.py`.
- Downloaded and standardized the Diabetic Retinopathy Debrecen dataset as `data/raw/dataset.csv`.
- Added `ucimlrepo` to the project dependencies.

### Changed
- Extended the Exercise 01 notebook to preserve LAB01 as the baseline and begin the LAB02 workflow.
- Updated the data-access process to avoid dependence on personal Google Drive paths.

### Documentation
- Added `docs/dataset_profile.md`.
- Documented the preliminary dataset-selection rationale.
- Added an academic-use disclaimer and a preliminary target-leakage assessment.

### Status
- Reproducible dataset access completed.
- Dataset schema audit, target definition, preprocessing, baseline comparison, and automated tests remain in progress.

## [1.0.0] - 2026-09-21

### Added
- Completed U01.LAB01 guided SVM laboratory using the Breast Cancer dataset.
- Implemented a leakage-safe SVM pipeline with StandardScaler and SVC.
- Added a DummyClassifier baseline for model comparison.
- Performed hyperparameter optimization using GridSearchCV and StratifiedKFold.
- Evaluated model performance using F1-Macro, ROC-AUC, classification reports, and confusion matrices.
- Added reusable build_svm() model factory in src/inf8239_u01/models.py.
- Added unit tests for model validation and pipeline integrity in tests/test_models.py.
- Exported cross-validation results and trained model artifacts.

### Deliverables
- notebooks/01_svm_guiada.ipynb
- src/inf8239_u01/models.py
- tests/test_models.py
- reports/svm_cv_results.csv
- reports/svm_best.joblib

## [0.1.0] - 2026-09-18

### Added
- Established the Unit 01 project structure (LAB00-LAB03 and Exercise_01-Exercise_02).
- Configured baseline environment dependencies in requirements.txt.
- Added .gitignore rules for virtual environments, cache directories, build artifacts, and raw datasets.
- Implemented environment verification modules and corresponding unit tests.
- Created the notebook 00_verificacion.ipynb for environment and interpreter validation.
- Established repository structure for reports, figures, and evidence generation.

### Validated
- Verified automated test execution using pytest.
- Confirmed reproducibility through a local virtual environment setup.
- Validated repository synchronization with the remote GitHub repository.

Overwriting /content/INF-8239-Ciencia-de-Datos-II/CHANGELOG.md


In [ ]:
# ------------------------------------------------------------------------------
# Synchronize the U02.LAB06 notebook with the repository
# ------------------------------------------------------------------------------

from google.colab import _message
from pathlib import Path
import json

# Destination path in the cloned repository
target = Path(
    "/content/INF-8239-Ciencia-de-Datos-II/"
    "Unit_02_Natural_Language_Processing/"
    "notebooks/"
    "U02.LAB06.ipynb"
)

# Verify that the destination folder exists
if not target.parent.is_dir():
    raise FileNotFoundError(
        f"The destination folder does not exist: {target.parent}"
    )

# Retrieve the currently open Colab notebook
nb = _message.blocking_request("get_ipynb")

# Save the notebook in the repository
with target.open("w", encoding="utf-8") as file:
    json.dump(
        nb["ipynb"],
        file,
        ensure_ascii=False,
        indent=1
    )

print("Notebook saved successfully to:")
print(target)

In [6]:
# ==============================================================================
# AUTOMATIC COMMIT AND AUTHENTICATED PUSH
# Objective: Preserve empty directories, stage repository changes, create a
# commit when needed, push the current branch securely, and verify Git status.
# ==============================================================================

# Import os for environment and file-system operations.
import os

# Import stat to configure the temporary authentication script permissions.
import stat

# Import subprocess to execute Git commands and verify their results.
import subprocess

# Import Path for operating-system-independent path management.
from pathlib import Path

# Import userdata to retrieve the GitHub token from Colab Secrets.
from google.colab import userdata


# ------------------------------------------------------------------------------
# 1. DEFINE THE REPOSITORY CONFIGURATION
# ------------------------------------------------------------------------------

# Define the repository root directory.
REPO_PATH = Path(
    "/content/INF-8239-Ciencia-de-Datos-II"
)

# Define the Unit 02 directory.
UNIT_02_PATH = (
    REPO_PATH
    / "Unit_02_Natural_Language_Processing"
)

# Define the GitHub repository information.
GITHUB_USER = "pascual-francisco"
GITHUB_REPO = "INF-8239-Ciencia-de-Datos-II"

# Define the commit message.
COMMIT_MESSAGE = "chore: sync repository updates"

# Retrieve the GitHub token from Colab Secrets.
GITHUB_TOKEN = userdata.get("TOKEN")

# Define the clean GitHub remote URL.
CLEAN_REMOTE_URL = (
    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
)


# ------------------------------------------------------------------------------
# 2. VALIDATE THE REPOSITORY AND TOKEN
# ------------------------------------------------------------------------------

# Confirm that the repository directory exists.
if not REPO_PATH.exists():
    raise FileNotFoundError(
        f"The repository was not found:\n{REPO_PATH}"
    )

# Confirm that the repository contains a .git directory.
if not (REPO_PATH / ".git").exists():
    raise FileNotFoundError(
        f"The directory is not a Git repository:\n{REPO_PATH}"
    )

# Confirm that the GitHub token is available.
if not GITHUB_TOKEN:
    raise ValueError(
        "The GitHub token was not found. "
        "Create a Colab Secret named TOKEN and enable notebook access."
    )


# ------------------------------------------------------------------------------
# 3. DEFINE A REUSABLE GIT COMMAND FUNCTION
# ------------------------------------------------------------------------------

def run_git(
    arguments: list[str],
    *,
    check: bool = True,
    capture_output: bool = False,
    environment: dict[str, str] | None = None
) -> subprocess.CompletedProcess:
    """
    Execute a Git command from the repository root.

    Parameters
    ----------
    arguments:
        Git arguments without the initial 'git'.
    check:
        Raise an error when the command returns a nonzero exit code.
    capture_output:
        Capture stdout and stderr instead of displaying them directly.
    environment:
        Optional environment variables used by the Git command.
    """

    return subprocess.run(
        ["git", *arguments],
        cwd=REPO_PATH,
        check=check,
        text=True,
        capture_output=capture_output,
        env=environment
    )


# ------------------------------------------------------------------------------
# 4. PREPARE EMPTY DIRECTORIES
# ------------------------------------------------------------------------------

print("=" * 70)
print("PREPARING EMPTY DIRECTORIES")
print("=" * 70)

if UNIT_02_PATH.exists():

    created_gitkeep_files = []

    # Walk through every directory inside Unit 02.
    for directory_path, directory_names, file_names in os.walk(
        UNIT_02_PATH
    ):
        # A directory is empty when it has no folders and no files.
        if not directory_names and not file_names:
            gitkeep_path = Path(directory_path) / ".gitkeep"
            gitkeep_path.touch()
            created_gitkeep_files.append(gitkeep_path)

    if created_gitkeep_files:
        print(
            f"Created {len(created_gitkeep_files)} "
            ".gitkeep file(s)."
        )
    else:
        print("No empty directories required a .gitkeep file.")

else:
    raise FileNotFoundError(
        f"The Unit 02 directory was not found:\n{UNIT_02_PATH}"
    )


# ------------------------------------------------------------------------------
# 5. DISPLAY THE CURRENT CHANGES
# ------------------------------------------------------------------------------

print("\n" + "=" * 70)
print("CHANGES DETECTED")
print("=" * 70)

# Display modified, deleted, renamed, and untracked files.
run_git([
    "status",
    "--short",
    "--untracked-files=all"
])


# ------------------------------------------------------------------------------
# 6. STAGE THE REPOSITORY CHANGES
# ------------------------------------------------------------------------------

# Stage all changes that are not excluded by .gitignore.
run_git([
    "add",
    "-A"
])

print("\n" + "=" * 70)
print("FILES STAGED FOR COMMIT")
print("=" * 70)

# Retrieve the exact list of staged changes.
staged_result = run_git(
    [
        "diff",
        "--cached",
        "--name-status"
    ],
    capture_output=True
)

staged_output = staged_result.stdout.strip()

# Display the staged changes.
if staged_output:
    print(staged_output)
else:
    print("No files are currently staged.")


# ------------------------------------------------------------------------------
# 7. CREATE A COMMIT ONLY WHEN CHANGES EXIST
# ------------------------------------------------------------------------------

if staged_output:
    print("\nCreating commit...")

    run_git([
        "commit",
        "-m",
        COMMIT_MESSAGE
    ])

    print(f"Commit created: {COMMIT_MESSAGE}")

else:
    print("\nNo new changes to commit.")


# ------------------------------------------------------------------------------
# 8. IDENTIFY THE CURRENT BRANCH
# ------------------------------------------------------------------------------

branch_result = run_git(
    [
        "branch",
        "--show-current"
    ],
    capture_output=True
)

current_branch = branch_result.stdout.strip()

if not current_branch:
    raise RuntimeError(
        "Git could not determine the current branch."
    )


# ------------------------------------------------------------------------------
# 9. CONFIGURE A CLEAN REMOTE URL
# ------------------------------------------------------------------------------

# Keep the permanent origin URL free from credentials.
run_git([
    "remote",
    "set-url",
    "origin",
    CLEAN_REMOTE_URL
])


# ------------------------------------------------------------------------------
# 10. CREATE TEMPORARY NON-INTERACTIVE AUTHENTICATION
# ------------------------------------------------------------------------------

# Define a temporary Git authentication helper.
askpass_path = Path("/tmp/github_colab_askpass.sh")

# Create a temporary script that responds to Git credential prompts.
askpass_content = """#!/bin/sh
case "$1" in
    *Username*) printf '%s\\n' "$GITHUB_USERNAME" ;;
    *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
    *) printf '\\n' ;;
esac
"""

# Save the authentication helper.
askpass_path.write_text(
    askpass_content,
    encoding="utf-8"
)

# Allow only the current user to execute or modify the helper.
askpass_path.chmod(
    stat.S_IRUSR
    | stat.S_IWUSR
    | stat.S_IXUSR
)

# Create an environment for the authenticated Git push.
push_environment = os.environ.copy()

# Tell Git which temporary authentication helper to use.
push_environment["GIT_ASKPASS"] = str(askpass_path)

# Prevent Git from opening an interactive credential prompt.
push_environment["GIT_TERMINAL_PROMPT"] = "0"

# Supply the GitHub username only through the process environment.
push_environment["GITHUB_USERNAME"] = GITHUB_USER

# Supply the GitHub token only through the process environment.
push_environment["GITHUB_TOKEN"] = GITHUB_TOKEN


# ------------------------------------------------------------------------------
# 11. PUSH THE CURRENT BRANCH
# ------------------------------------------------------------------------------

print("\n" + "=" * 70)
print("PUSHING TO GITHUB")
print("=" * 70)
print(f"Current branch: {current_branch}")

try:
    # Push the current branch and establish its upstream relationship.
    run_git(
        [
            "push",
            "--set-upstream",
            "origin",
            current_branch
        ],
        environment=push_environment
    )

finally:
    # Delete the temporary authentication helper after the push attempt.
    if askpass_path.exists():
        askpass_path.unlink()


# ------------------------------------------------------------------------------
# 12. DISPLAY THE LATEST COMMITS
# ------------------------------------------------------------------------------

print("\n" + "=" * 70)
print("LATEST COMMITS")
print("=" * 70)

# Display the five most recent commits.
run_git([
    "log",
    "--oneline",
    "--decorate",
    "-5"
])


# ------------------------------------------------------------------------------
# 13. VERIFY THE FINAL SYNCHRONIZATION STATUS
# ------------------------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL SYNCHRONIZATION STATUS")
print("=" * 70)

# Display the final branch and working-tree status.
run_git([
    "status",
    "--short",
    "--branch"
])

print("\nRepository synchronized successfully.")

PREPARING EMPTY DIRECTORIES
No empty directories required a .gitkeep file.

CHANGES DETECTED

FILES STAGED FOR COMMIT
No files are currently staged.

No new changes to commit.

PUSHING TO GITHUB
Current branch: main

LATEST COMMITS

FINAL SYNCHRONIZATION STATUS

Repository synchronized successfully.
